# Data analysis

*Here are the possible analysis steps. **The order and choice depend on the needs of your data**, which you explored in the previous tutorial.*

## 1. Label traces by masking
Give a label for each trace that match with the mask in order to make both population of traces.

In [ ]:
!trace_assign_mask --input trace_file.ecsv --mask_file my_mask.npy --label mymask

## 2. Correct Z coordinates
Adjust localization coordinates along the Z-axis, registration based on the center of mass of each trace.

In [ ]:
!trace_correct_coordinates --input traces.ecsv --output traces_corrected.ecsv --max_iter 10 --tolerance 0.01

## 3. Impute genomic coordinates
Enhance data information by adding genomic coordinates.


In [ ]:
!trace_genomic_coordinates --input trace_file.ecsv --bed bed_file.bed --output output_file.ecsv

## 4. Direct filtering
There are multiple way to filter your traces:
1. Spatial coordinates (x, y, z)
2. Minimum number of barcodes per trace
3. Remove a specific barcode
4. Label-based filtering : remove or keep traces with a specific label
5. Localization intensity thresholding
6. Duplicate spot removal :
   - If you provide a localization file: keep the brightness;
   - Else remove both and remove barcodes with same UID (I don't when it's happen, wrong file manipulation ?).

In [ ]:
# Basic filtering with spatial constraints and minimum barcode requirement
$ trace_filter --input Trace.ecsv --z_min 4 --z_max 5 --y_max 175 --output zy_filtered --n_barcodes 3

# Remove duplicate spots and specific barcodes
$ trace_filter --input Trace.ecsv --clean_spots --remove_barcode 1,3,5

# Keep only traces with a specific label
$ trace_filter --input Trace.ecsv --keep_label region1

# Remove traces with a specific label
$ trace_filter --input Trace.ecsv --remove_label region1

# Filter by localization intensity
$ trace_filter --input Trace.ecsv --localization_file Localizations.ecsv --intensity_min 1000

# Process multiple files via pipe
$ ls *Trace.ecsv | trace_filter --pipe --n_barcodes 3

## 5. Advanced filtering
This script aim to filter your traces by applying different analysis steps at same time:
1. For each trace, make a KDTree clustering to split trace if necessary and to remove outliers localizations.
2. Duplicate management:
    - if two localisations are very similar, create an intermediate localisation;
    - otherwise, delete all duplicates in the trace.

In [ ]:
usage: trace_filter_advanced [-h] [-F ROOTFOLDER] [-O OUTPUT] [--input INPUT]
                             [--overlapping_threshold OVERLAPPING_THRESHOLD]
                             [--N_barcodes N_BARCODES] [--pipe]

## 6. Split aggregated traces
1. Computes the **3D radius of gyration** (Rg) for each trace
2. Computes a threshold based on the parameter "std_threshold" : Number of standard deviations above mean Rg to classify as large. (`threshold = mean_rg + std_threshold * std_rg`)
3. If a trace exceeds this threshold: **K-means clustering** (default K=2), the trace is replaced by 2 new traces.

In [ ]:
!trace_splitter --input {data_path}/dest/merged_traces_filtered.ecsv